In [2]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import KFold
# import xgboost as xgb
from scipy import interpolate
from scipy.stats import linregress
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ------------------------------
# Configuration
# ------------------------------
DATA_ROOT = "../../Data"        # Folder containing S1, S2, ...
GRADES_CSV = "../../outputs/grades_wide.csv"      # CSV with columns: student_id,final,midterm_1,midterm_2

TARGET_FS = 4.0
WINDOW_SEC = 60
STEP_SEC = 30
BINARY_THRESHOLD = 70.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 30  # Reduced for cross-validation
LR = 1e-3
N_FOLDS = 10  # 10-fold cross-validation

# ------------------------------
# 1. Load grades
# ------------------------------
def load_grades(csv_path):
    df = pd.read_csv(csv_path)
    def normalize_id(sid):
        sid = str(sid).upper().strip()
        match = re.search(r'S(\d+)', sid)
        if match:
            num = int(match.group(1))
            return f"S{num}"
        return sid
    
    df['student_id'] = df['student_id'].apply(normalize_id)
    df['final'] = df['final'] / 2.0
    
    grades_dict = {
        'Midterm 1': dict(zip(df['student_id'], df['midterm_1'])),
        'Midterm 2': dict(zip(df['student_id'], df['midterm_2'])),
        'Final': dict(zip(df['student_id'], df['final']))
    }
    return grades_dict

grades_dict = load_grades(GRADES_CSV)

# ------------------------------
# 2. Load multimodal signals
# ------------------------------
def load_signal(csv_path, cols=None):
    df = pd.read_csv(csv_path, header=None)
    start_time = df.iloc[0, 0]
    fs = df.iloc[1, 0]
    data = df.iloc[2:].values.astype(float)
    if cols is not None:
        data = data[:, cols]
    if data.ndim == 1:
        data = data[:, None]
    n_samples = data.shape[0]
    timestamps = start_time + np.arange(n_samples) / fs
    return timestamps, data, fs

def resample_signal(timestamps, data, orig_fs, target_fs=TARGET_FS):
    t_start, t_end = timestamps[0], timestamps[-1]
    new_t = np.arange(t_start, t_end, 1.0/target_fs)
    resampled = np.zeros((len(new_t), data.shape[1]))
    for ch in range(data.shape[1]):
        f = interpolate.interp1d(timestamps, data[:, ch], kind='linear', fill_value='extrapolate')
        resampled[:, ch] = f(new_t)
    return new_t, resampled

def load_session(subj_folder, exam_name):
    exam_path = os.path.join(subj_folder, exam_name)
    if not os.path.exists(exam_path):
        return None
    
    signals = {}
    mods = {
        'ACC': ('ACC.csv', [0,1,2]),
        'BVP': ('BVP.csv', 0),
        'EDA': ('EDA.csv', 0),
        'HR': ('HR.csv', 0),
        'TEMP': ('TEMP.csv', 0)
    }
    
    for name, (fname, cols) in mods.items():
        path = os.path.join(exam_path, fname)
        if not os.path.exists(path):
            continue
        try:
            ts, data, fs = load_signal(path, cols)
            if data.size == 0:
                continue
            new_ts, resampled = resample_signal(ts, data, fs, TARGET_FS)
            signals[name] = (new_ts, resampled)
        except Exception as e:
            continue
    
    ibi_path = os.path.join(exam_path, "IBI.csv")
    if os.path.exists(ibi_path):
        try:
            df_ibi = pd.read_csv(ibi_path, header=None)
            start_time = df_ibi.iloc[0, 0]
            ibi_vals = df_ibi.iloc[2:, 0].values.astype(float)
            ibi_times = start_time + np.cumsum(ibi_vals)
            signals['IBI_times'] = ibi_times
            signals['IBI_vals'] = ibi_vals
        except:
            signals['IBI_times'] = np.array([])
            signals['IBI_vals'] = np.array([])
    else:
        signals['IBI_times'] = np.array([])
        signals['IBI_vals'] = np.array([])
    
    subj_name = os.path.basename(subj_folder)
    match = re.search(r'S(\d+)', subj_name.upper())
    if match:
        subj_id = f"S{int(match.group(1))}"
    else:
        subj_id = subj_name.upper()
    
    grade = grades_dict[exam_name].get(subj_id, None)
    if grade is None:
        return None
    
    signals['grade'] = grade
    signals['binary_label'] = 1.0 if grade < BINARY_THRESHOLD else 0.0
    signals['subject_id'] = subj_id
    
    return signals

# ------------------------------
# 3. Window extraction
# ------------------------------
def compute_hrv_features(ibi_times, ibi_vals, window_start, window_end):
    mask = (ibi_times >= window_start) & (ibi_times <= window_end)
    ibi_in_win = ibi_vals[mask]
    if len(ibi_in_win) < 2:
        return [0.0, 0.0, 0.0]
    diff = np.diff(ibi_in_win)
    rmssd = np.sqrt(np.mean(diff**2))
    sdnn = np.std(ibi_in_win)
    mean_hr = 60.0 / np.mean(ibi_in_win)
    return [rmssd, sdnn, mean_hr]

def extract_windows(signals, window_sec, step_sec, fs=TARGET_FS):
    if 'ACC' not in signals:
        return [], [], []
    
    t_acc, acc_data = signals['ACC']
    win_len = int(window_sec * fs)
    step_len = int(step_sec * fs)
    
    mod_list = ['ACC', 'BVP', 'EDA', 'HR', 'TEMP']
    raw_data_list = []
    for mod in mod_list:
        if mod in signals:
            _, data = signals[mod]
            raw_data_list.append(data)
    
    if not raw_data_list:
        return [], [], []
    
    min_len = min(d.shape[0] for d in raw_data_list)
    raw_data = np.concatenate([d[:min_len] for d in raw_data_list], axis=1)
    t_axis = t_acc[:min_len]
    
    ibi_times = signals.get('IBI_times', np.array([]))
    ibi_vals = signals.get('IBI_vals', np.array([]))
    
    static_feats = []
    sequences = []
    labels = []
    
    for start_idx in range(0, min_len - win_len + 1, step_len):
        end_idx = start_idx + win_len
        win_t_start = t_axis[start_idx]
        win_t_end = t_axis[end_idx-1]
        X_raw = raw_data[start_idx:end_idx, :]
        
        static = []
        for ch in range(X_raw.shape[1]):
            mean_val = np.mean(X_raw[:, ch])
            std_val = np.std(X_raw[:, ch])
            min_val = np.min(X_raw[:, ch])
            max_val = np.max(X_raw[:, ch])
            x_vals = np.arange(X_raw.shape[0])
            slope, _, _, _, _ = linregress(x_vals, X_raw[:, ch])
            static.extend([mean_val, std_val, min_val, max_val, slope])
        
        hrv = compute_hrv_features(ibi_times, ibi_vals, win_t_start, win_t_end)
        static.extend(hrv)
        
        static_feats.append(np.array(static, dtype=np.float32))
        sequences.append(X_raw.astype(np.float32))
        labels.append(signals['binary_label'])
    
    return static_feats, sequences, labels

# ------------------------------
# 4. Load all data
# ------------------------------
participants = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)) and d.upper().startswith('S')]
participants.sort()

all_samples = []

for subj_folder in participants:
    subj_path = os.path.join(DATA_ROOT, subj_folder)
    for exam in ['Midterm 1', 'Midterm 2', 'Final']:
        signals = load_session(subj_path, exam)
        if signals is None:
            continue
        static_list, seq_list, label_list = extract_windows(signals, WINDOW_SEC, STEP_SEC, TARGET_FS)
        for s, seq, lab in zip(static_list, seq_list, label_list):
            all_samples.append({
                'static': s,
                'seq': seq,
                'label': lab,
                'subject': signals['subject_id']
            })

print(f"Total samples: {len(all_samples)}")

# Extract features and labels for cross-validation
X_static = np.array([s['static'] for s in all_samples])
X_sequences = [s['seq'] for s in all_samples]
y = np.array([s['label'] for s in all_samples])

# Standardize static features
scaler = StandardScaler()
X_static_scaled = scaler.fit_transform(X_static)

# ------------------------------
# 5. Deep Learning Models with detailed epoch logging
# ------------------------------
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        X = torch.tensor(self.sequences[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return X, y

class LSTMWithoutAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        out = h_n[-1]
        return self.classifier(out).squeeze()

class GRUWithoutAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        _, h_n = self.gru(x)
        out = h_n[-1]
        return self.classifier(out).squeeze()

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_out):
        score = self.v(torch.tanh(self.W(lstm_out)))
        attn_weights = torch.softmax(score, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context, attn_weights

class LSTMWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.attention = Attention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context, attn_weights = self.attention(lstm_out)
        return self.classifier(context).squeeze(), attn_weights

class GRUWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.attention = Attention(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        gru_out, _ = self.gru(x)
        context, attn_weights = self.attention(gru_out)
        return self.classifier(context).squeeze(), attn_weights

def train_dl_model_with_logging(model, train_loader, val_loader, test_loader, model_name, fold, epochs=EPOCHS, lr=LR):
    """Train model with detailed logging after each epoch"""
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'test_loss': [], 'test_acc': []
    }
    
    best_val_loss = float('inf')
    best_model_state = None
    
    print(f"\n{'='*80}")
    print(f"Fold {fold+1} - Training {model_name}")
    print(f"{'='*80}")
    print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12} {'Test Loss':<12} {'Test Acc':<12}")
    print(f"{'-'*80}")
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            
            if hasattr(model, 'attention'):
                output, _ = model(X)
            else:
                output = model(X)
            
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            probs = torch.sigmoid(output)
            preds = (probs > 0.5).float()
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(y.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = accuracy_score(train_labels, train_preds)
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        
        # Validation
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                if hasattr(model, 'attention'):
                    output, _ = model(X)
                else:
                    output = model(X)
                loss = criterion(output, y)
                val_loss += loss.item()
                probs = torch.sigmoid(output)
                preds = (probs > 0.5).float()
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        # Test evaluation
        model.eval()
        test_loss = 0
        test_preds = []
        test_labels = []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                if hasattr(model, 'attention'):
                    output, _ = model(X)
                else:
                    output = model(X)
                loss = criterion(output, y)
                test_loss += loss.item()
                probs = torch.sigmoid(output)
                preds = (probs > 0.5).float()
                test_preds.extend(preds.cpu().numpy())
                test_labels.extend(y.cpu().numpy())
        
        avg_test_loss = test_loss / len(test_loader)
        test_acc = accuracy_score(test_labels, test_preds)
        history['test_loss'].append(avg_test_loss)
        history['test_acc'].append(test_acc)
        
        # Print epoch results
        print(f"{epoch+1:<6} {avg_train_loss:<12.4f} {train_acc:<12.4f} {avg_val_loss:<12.4f} {val_acc:<12.4f} {avg_test_loss:<12.4f} {test_acc:<12.4f}")
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    final_metrics = evaluate_dl_model(model, test_loader, return_attn=False)
    
    return model, history, final_metrics

def evaluate_dl_model(model, test_loader, return_attn=False):
    model.eval()
    all_preds = []
    all_labels = []
    all_attn_weights = []
    
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if hasattr(model, 'attention'):
                output, attn = model(X)
                if return_attn:
                    all_attn_weights.append(attn.cpu().numpy())
            else:
                output = model(X)
            
            probs = torch.sigmoid(output)
            preds = (probs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_preds)
    
    metrics = {'accuracy': acc, 'f1': f1, 'auc': auc}
    
    if return_attn and all_attn_weights:
        return metrics, all_attn_weights
    return metrics

# ------------------------------
# 6. Traditional ML with cross-validation
# ------------------------------
def train_eval_ml_cv(model_class, model_name, params, X, y, n_folds=N_FOLDS):
    """Train ML model with cross-validation and detailed logging"""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    cv_scores = {'accuracy': [], 'f1': [], 'auc': []}
    fold_results = []
    
    print(f"\n{'='*80}")
    print(f"Training {model_name} with {n_folds}-fold CV")
    print(f"{'='*80}")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Train model
        model = model_class(**params)
        model.fit(X_train, y_train)
        
        # Predictions
        y_pred = model.predict(X_val)
        
        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        auc = roc_auc_score(y_val, y_pred)
        
        cv_scores['accuracy'].append(acc)
        cv_scores['f1'].append(f1)
        cv_scores['auc'].append(auc)
        fold_results.append((fold, acc, f1, auc))
        
        print(f"Fold {fold+1}: Acc={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    
    # Summary
    print(f"\n{model_name} - Cross-Validation Results:")
    print(f"  Accuracy: {np.mean(cv_scores['accuracy']):.4f} ± {np.std(cv_scores['accuracy']):.4f}")
    print(f"  F1 Score:  {np.mean(cv_scores['f1']):.4f} ± {np.std(cv_scores['f1']):.4f}")
    print(f"  AUC:       {np.mean(cv_scores['auc']):.4f} ± {np.std(cv_scores['auc']):.4f}")
    
    return cv_scores, fold_results

# ------------------------------
# 7. Deep Learning with cross-validation
# ------------------------------
def run_dl_cv(model_class, model_name, input_dim, X_sequences, y, n_folds=N_FOLDS):
    """Run deep learning with cross-validation and detailed epoch logging"""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    all_fold_metrics = []
    all_fold_histories = []
    
    print(f"\n{'#'*80}")
    print(f"Deep Learning Model: {model_name}")
    print(f"{'#'*80}")
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_sequences)):
        # For DL, we further split train into train/val (80/20)
        train_indices = train_idx
        val_size = int(0.2 * len(train_indices))
        val_indices = train_indices[:val_size]
        train_indices = train_indices[val_size:]
        
        # Create datasets
        train_dataset = Subset(SequenceDataset(X_sequences, y), train_indices)
        val_dataset = Subset(SequenceDataset(X_sequences, y), val_indices)
        test_dataset = Subset(SequenceDataset(X_sequences, y), test_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        # Initialize model
        model = model_class(input_dim)
        
        # Train with logging
        trained_model, history, metrics = train_dl_model_with_logging(
            model, train_loader, val_loader, test_loader, model_name, fold
        )
        
        all_fold_metrics.append(metrics)
        all_fold_histories.append(history)
    
    # Aggregate results
    avg_metrics = {
        'accuracy': np.mean([m['accuracy'] for m in all_fold_metrics]),
        'accuracy_std': np.std([m['accuracy'] for m in all_fold_metrics]),
        'f1': np.mean([m['f1'] for m in all_fold_metrics]),
        'f1_std': np.std([m['f1'] for m in all_fold_metrics]),
        'auc': np.mean([m['auc'] for m in all_fold_metrics]),
        'auc_std': np.std([m['auc'] for m in all_fold_metrics])
    }
    
    print(f"\n{model_name} - Cross-Validation Summary:")
    print(f"  Accuracy: {avg_metrics['accuracy']:.4f} ± {avg_metrics['accuracy_std']:.4f}")
    print(f"  F1 Score: {avg_metrics['f1']:.4f} ± {avg_metrics['f1_std']:.4f}")
    print(f"  AUC: {avg_metrics['auc']:.4f} ± {avg_metrics['auc_std']:.4f}")
    
    return avg_metrics, all_fold_histories

# ------------------------------
# 8. Run all experiments
# ------------------------------
print("="*80)
print("STRESS/ANXIETY DETECTION - COMPREHENSIVE EVALUATION")
print("="*80)

# Traditional ML Models
ml_configs = {
    'Decision Tree': (DecisionTreeClassifier, {'max_depth': 5, 'random_state': 42}),
    'SVM (RBF)': (SVC, {'kernel': 'rbf', 'probability': True, 'random_state': 42}),
    'Random Forest': (RandomForestClassifier, {'n_estimators': 100, 'random_state': 42}),
    # 'XGBoost': (xgb.XGBClassifier, {'n_estimators': 100, 'learning_rate': 0.1, 'random_state': 42})
}

ml_results = {}
for name, (model_class, params) in ml_configs.items():
    cv_scores, fold_results = train_eval_ml_cv(model_class, name, params, X_static_scaled, y, N_FOLDS)
    ml_results[name] = {
        'accuracy': np.mean(cv_scores['accuracy']),
        'accuracy_std': np.std(cv_scores['accuracy']),
        'f1': np.mean(cv_scores['f1']),
        'f1_std': np.std(cv_scores['f1']),
        'auc': np.mean(cv_scores['auc']),
        'auc_std': np.std(cv_scores['auc'])
    }

# Deep Learning Models
input_dim = X_sequences[0].shape[1]
print(f"\nInput feature dimension for DL models: {input_dim}")

dl_configs = {
    'LSTM': LSTMWithoutAttention,
    'GRU': GRUWithoutAttention,
    'LSTM+Attention': LSTMWithAttention,
    'GRU+Attention': GRUWithAttention
}

dl_results = {}
dl_histories = {}

for name, model_class in dl_configs.items():
    avg_metrics, histories = run_dl_cv(model_class, name, input_dim, X_sequences, y, N_FOLDS)
    dl_results[name] = avg_metrics
    dl_histories[name] = histories

# ------------------------------
# 9. Results Comparison and Visualization
# ------------------------------
print("\n" + "="*80)
print("FINAL RESULTS COMPARISON")
print("="*80)

# Create comparison table
all_results = {**ml_results, **dl_results}
comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df[['accuracy', 'accuracy_std', 'f1', 'f1_std', 'auc', 'auc_std']]
comparison_df = comparison_df.sort_values('accuracy', ascending=False)

print("\nModel Performance (mean ± std):")
print(comparison_df.to_string())

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Accuracy comparison with error bars
models = comparison_df.index.tolist()
acc_mean = comparison_df['accuracy'].values
acc_std = comparison_df['accuracy_std'].values
f1_mean = comparison_df['f1'].values
f1_std = comparison_df['f1_std'].values
auc_mean = comparison_df['auc'].values
auc_std = comparison_df['auc_std'].values

y_pos = np.arange(len(models))

# Accuracy plot
axes[0, 0].barh(y_pos, acc_mean, xerr=acc_std, capsize=5, color='skyblue', edgecolor='black')
axes[0, 0].set_yticks(y_pos)
axes[0, 0].set_yticklabels(models)
axes[0, 0].set_xlabel('Accuracy')
axes[0, 0].set_title(f'Accuracy Comparison ({N_FOLDS}-fold CV)')
axes[0, 0].set_xlim(0, 1)
axes[0, 0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
axes[0, 0].grid(True, alpha=0.3)

# F1 Score plot
axes[0, 1].barh(y_pos, f1_mean, xerr=f1_std, capsize=5, color='lightgreen', edgecolor='black')
axes[0, 1].set_yticks(y_pos)
axes[0, 1].set_yticklabels(models)
axes[0, 1].set_xlabel('F1 Score')
axes[0, 1].set_title('F1 Score Comparison')
axes[0, 1].set_xlim(0, 1)
axes[0, 1].grid(True, alpha=0.3)

# AUC plot
axes[1, 0].barh(y_pos, auc_mean, xerr=auc_std, capsize=5, color='salmon', edgecolor='black')
axes[1, 0].set_yticks(y_pos)
axes[1, 0].set_yticklabels(models)
axes[1, 0].set_xlabel('AUC-ROC')
axes[1, 0].set_title('AUC-ROC Comparison')
axes[1, 0].set_xlim(0, 1)
axes[1, 0].grid(True, alpha=0.3)

# Training curves for best deep learning model
best_dl_model = comparison_df[comparison_df.index.isin(dl_results.keys())].iloc[0].name
print(f"\nBest performing DL model: {best_dl_model}")

if best_dl_model in dl_histories:
    # Plot training curves for first fold of best model
    best_history = dl_histories[best_dl_model][0]
    epochs = range(1, len(best_history['train_loss']) + 1)
    
    axes[1, 1].plot(epochs, best_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[1, 1].plot(epochs, best_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[1, 1].plot(epochs, best_history['test_loss'], 'g-', label='Test Loss', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].set_title(f'{best_dl_model} - Training Curves (Fold 1)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Create second figure for accuracy curves
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    ax2.plot(epochs, best_history['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
    ax2.plot(epochs, best_history['val_acc'], 'r-', label='Val Accuracy', linewidth=2)
    ax2.plot(epochs, best_history['test_acc'], 'g-', label='Test Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{best_dl_model} - Accuracy Curves (Fold 1)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{best_dl_model}_accuracy_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plt.tight_layout()
plt.savefig('model_comparison_cv.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------
# 10. Detailed Metrics Summary
# ------------------------------
print("\n" + "="*80)
print("DETAILED METRICS SUMMARY")
print("="*80)

# Create summary text file
with open('experiment_results_summary.txt', 'w') as f:
    f.write("STRESS/ANXIETY DETECTION - EXPERIMENT RESULTS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Dataset Statistics:\n")
    f.write(f"  Total samples: {len(all_samples)}\n")
    f.write(f"  Class distribution:\n")
    unique, counts = np.unique(y, return_counts=True)
    for cls, cnt in zip(unique, counts):
        stress_level = "High Stress" if cls == 1 else "Low Stress"
        f.write(f"    {stress_level}: {cnt} ({cnt/len(y)*100:.1f}%)\n")
    f.write(f"\nCross-Validation: {N_FOLDS}-fold\n\n")
    
    f.write("Model Performance (mean ± std):\n")
    f.write("-"*80 + "\n")
    for model in comparison_df.index:
        f.write(f"\n{model}:\n")
        f.write(f"  Accuracy: {comparison_df.loc[model, 'accuracy']:.4f} ± {comparison_df.loc[model, 'accuracy_std']:.4f}\n")
        f.write(f"  F1 Score: {comparison_df.loc[model, 'f1']:.4f} ± {comparison_df.loc[model, 'f1_std']:.4f}\n")
        f.write(f"  AUC:      {comparison_df.loc[model, 'auc']:.4f} ± {comparison_df.loc[model, 'auc_std']:.4f}\n")

print("\n✅ All experiments completed successfully!")
print(f"Results saved to:")
print("  - model_comparison_cv.png (comparison bar charts)")
print("  - *_accuracy_curves.png (training curves for best model)")
print("  - experiment_results_summary.txt (detailed results)")

Total samples: 14724
STRESS/ANXIETY DETECTION - COMPREHENSIVE EVALUATION

Training Decision Tree with 10-fold CV
Fold 1: Acc=0.7522, F1=0.4122, AUC=0.6211
Fold 2: Acc=0.7604, F1=0.4770, AUC=0.6479
Fold 3: Acc=0.7794, F1=0.6181, AUC=0.7289
Fold 4: Acc=0.7807, F1=0.5444, AUC=0.6842
Fold 5: Acc=0.7615, F1=0.5725, AUC=0.6971
Fold 6: Acc=0.7677, F1=0.5899, AUC=0.7102
Fold 7: Acc=0.7575, F1=0.4020, AUC=0.6147
Fold 8: Acc=0.7670, F1=0.5189, AUC=0.6676
Fold 9: Acc=0.7880, F1=0.5884, AUC=0.7070
Fold 10: Acc=0.7466, F1=0.4618, AUC=0.6378

Decision Tree - Cross-Validation Results:
  Accuracy: 0.7661 ± 0.0126
  F1 Score:  0.5185 ± 0.0730
  AUC:       0.6716 ± 0.0379

Training SVM (RBF) with 10-fold CV
Fold 1: Acc=0.7848, F1=0.5318, AUC=0.6770
Fold 2: Acc=0.7936, F1=0.5607, AUC=0.6921
Fold 3: Acc=0.8052, F1=0.5509, AUC=0.6888
Fold 4: Acc=0.7977, F1=0.5300, AUC=0.6782
Fold 5: Acc=0.7908, F1=0.5333, AUC=0.6787
Fold 6: Acc=0.7914, F1=0.5465, AUC=0.6854
Fold 7: Acc=0.7982, F1=0.5075, AUC=0.6678
Fold 8:

ValueError: Target size (torch.Size([1])) must be the same as input size (torch.Size([]))